# Neural Network-Driven MIMO-OFDM Receiver

This notebook contains the main implementation used in the project.

**Project highlights**
- 2×2 MIMO-OFDM communication system
- NVIDIA Sionna / TensorFlow
- LDPC and QAM physical-layer processing
- Neural-network-based receiver
- SDR-based OTA experiment

> Some project files and receiver-related modules are intentionally not included in the public repository.


## 1. Environment Setup

Library imports, GPU configuration, and runtime environment setup.


In [ ]:
import os  # Configure which GPU 
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0  # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
print("--------------------------------------------------------------------------------------------------------------------------------------------------------")
# Import Sionna
try:
    import sionna as sn
    print("Sionna version:", sn.__version__)
except ImportError as e:
    # Install Sionna if package is not already installed
    import os
    os.system("pip install sionna")
    import sionna as sn
    print("Sionna installed and imported. Version:", sn.__version__)

# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print("Memory growth set for GPU")
    except RuntimeError as e:
        print(e)

# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

import numpy as np
print("NumPy version:", np.__version__)

# For saving complex Python data structures efficiently
import pickle
print("pickle imported successfully")

# For plotting
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib
print("Matplotlib version:", matplotlib.__version__)
import time
import random
from scipy.signal import convolve
from scipy.signal import lfilter
from scipy.signal import correlate
from scipy.io import loadmat

# For the implementation of the neural receiver
from tensorflow.keras import Model
from tensorflow.keras.layers import Layer, Conv2D, LayerNormalization
from tensorflow.nn import relu
print("Keras Model and layers imported successfully")


## 2. MIMO-OFDM System Configuration

Communication-system parameters, resource-grid configuration, antenna setup, channel model, and LDPC encoder/decoder configuration.


In [ ]:
# Bit per channel use
NUM_BITS_PER_SYMBOL = 4 # 16 QAM

# Minimum value of Eb/N0 [dB] for simulations
EBN0_DB_MIN = 10.0

# Maximum value of Eb/N0 [dB] for simulations
EBN0_DB_MAX = 40.0

# How many examples are processed by Sionna in parallel
BATCH_SIZE = 16 # 16

# Coding rate
CODERATE = 0.5

# Define the number of UT and BS antennas
NUM_UT = 1
NUM_BS = 1
NUM_UT_ANT = 2
NUM_BS_ANT = 2

# The number of transmitted streams is equal to the number of UT antennas
# in both uplink and downlink
#NUM_STREAMS_PER_TX = NUM_UT_ANT
NUM_STREAMS_PER_TX = NUM_UT_ANT

RX_TX_ASSOCIATION = np.array([[1]]) # 接收器 0 從發射器 0 和 1 接收數據


# Instantiate a StreamManagement object
# This determines which data streams are determined for which receiver.
# In this simple setup, this is fairly easy. However, it can get more involved
# for simulations with many transmitters and receivers.
STREAM_MANAGEMENT = sn.phy.mimo.StreamManagement(RX_TX_ASSOCIATION, NUM_STREAMS_PER_TX)

Cyclic_Prefix_Length = 144
FFT_size = 2048
Num_Ofdm_Symbols = 14
# pilot 的位置

pilot_ofdm_symbol_indices_loop = []
    
RESOURCE_GRID = sn.phy.ofdm.ResourceGrid( num_ofdm_symbols=Num_Ofdm_Symbols,
                                      fft_size=FFT_size,
                                      subcarrier_spacing=60e3,
                                      num_tx=NUM_UT,
                                      num_streams_per_tx=NUM_STREAMS_PER_TX,
                                      cyclic_prefix_length=Cyclic_Prefix_Length,
                                      num_guard_carriers=[202,201],
                                      dc_null=True,
                                      pilot_pattern="kronecker",
                                      pilot_ofdm_symbol_indices=[2, 11])


# Carrier frequency in Hz.
CARRIER_FREQUENCY = 3.8e9

# Antenna setting
UT_ARRAY = sn.phy.channel.tr38901.AntennaArray(  num_rows=1,
                                        num_cols=int(NUM_UT_ANT),
                                        polarization="single",
                                        polarization_type="V",
                                        antenna_pattern="38.901",
                                        carrier_frequency=CARRIER_FREQUENCY)
BS_ARRAY = sn.phy.channel.tr38901.AntennaArray( num_rows=1,  
                                            num_cols=int(NUM_BS_ANT / 2 ),
                                            polarization="dual",
                                            polarization_type="cross",
                                            antenna_pattern="38.901", # Try 'omni'
                                            carrier_frequency=CARRIER_FREQUENCY)

# en、decode
encoder = sn.phy.fec.ldpc.LDPC5GEncoder(1644, 1644*2)
decoder = sn.phy.fec.ldpc.LDPC5GDecoder(encoder, hard_out=True,num_iter=5)

# 開啟隨機排序
shuffle_indices = np.load(fr"params/shuffle_indices.npy")

# Nominal delay spread in [s]. Please see the CDL documentation
# about how to choose this value.
DELAY_SPREAD = 100e-9

# The `direction` determines if the UT or BS is transmitting.
# In the `uplink`, the UT is transmitting.
DIRECTION = "uplink"

# Suitable values are ["A", "B", "C", "D", "E"]
CDL_MODEL = "C"

# UT speed [m/s]. BSs are always assumed to be fixed.
# The direction of travel will chosen randomly within the x-y plane.
SPEED = 0.0

# Configure a channel impulse reponse (CIR) generator for the CDL model.
CDL = sn.phy.channel.tr38901.CDL(CDL_MODEL,
                             DELAY_SPREAD,
                             CARRIER_FREQUENCY,
                             UT_ARRAY,
                             BS_ARRAY,
                             DIRECTION,
                             min_speed=SPEED)
print("--------------------------------------------------------------------------------------------------------------------------------------------------------")

# QAM modulation order (bits per symbol)
print(2 ** NUM_BITS_PER_SYMBOL,"QAM")

# Carrier frequency
print("Carrier frequency :", CARRIER_FREQUENCY/(1e9),"(GHz)")

# Code rate
print("LDPC 碼率:", CODERATE)

# Number of antennas
print("TX 數量 :", NUM_UT_ANT)
print("RX 數量 :", NUM_BS_ANT)
print("--------------------------------------------------------------------------------------------------------------------------------------------------------")

## 3. Receiver and Demonstration Interface

Receiver execution, BER/SNR evaluation, reconstructed-image display, and GUI workflow.

> **Note:** This section depends on project modules and assets that may not be included in the public repository.


In [ ]:
import threading
import tkinter as tk
import time
from PIL import Image, ImageTk
import numpy as np

from reciver import EP_detector

# 設置環境變數，確保 WSL GUI 能夠顯示中文

class App:
    def __init__(self, root):
        self.iqtimes = 1
        self.root = root
        self.root.title("GUI")
        self.root.geometry("1000x600")  # 設定視窗大小

        self.running = False  # 控制程序是否運行
        self.lock = threading.Lock()  # 避免多執行緒問題
        
        # 期望的數據形狀
        self.expected_shape = (2, 1227520)
        
        # 加載原始數據
        try:
            self.orign_bit = np.load("params/Picture_bits_monkey.npy")
        except Exception as e:
            print(f"❌ 加載 Picture_bits_monkey.npy 失敗: {e}")
            self.orign_bit = np.random.randint(0, 2, (1, 3024600))
        
        try:
            self.ori_picture = np.load("params/ori_picture.npy")
        except Exception as e:
            print(f"❌ 加載 ori_picture.npy 失敗: {e}")
            self.ori_picture = np.zeros((355, 355, 3), dtype=np.uint8)

        # 初始化 BER 和 SNR 數據
        # self.ber_NRX_values = []
        self.ber_EP_values = []
        self.snr_values = []
        
        # 載入檢測器
        # self.NRX_DETECTOR = NRX_detector()
        self.EP_DETECTOR = EP_detector()

        # Tkinter 圖像初始化
        self.init_gui()

    def init_gui(self):
        """初始化 Tkinter GUI 組件"""
        self.tk_img = ImageTk.PhotoImage(Image.fromarray(np.zeros((250, 250, 3), dtype=np.uint8)))
        # self.tk_img2 = ImageTk.PhotoImage(Image.fromarray(np.zeros((250, 250, 3), dtype=np.uint8)))
        self.tk_img_original = ImageTk.PhotoImage(Image.fromarray(self.ori_picture).resize((250, 250), Image.LANCZOS))

        # 原始圖片
        tk.Label(self.root, text="ori picture").grid(row=0, column=0, padx=10, pady=10)
        self.original_image_label = tk.Label(self.root, image=self.tk_img_original)
        self.original_image_label.grid(row=1, column=0, padx=10, pady=10)

        # EP
        tk.Label(self.root, text="EP Detector").grid(row=0, column=1, padx=10, pady=10)
        self.reconstructed_image_label = tk.Label(self.root, image=self.tk_img)
        self.reconstructed_image_label.grid(row=1, column=1, padx=10, pady=10)

        # NRX
        # tk.Label(self.root, text="NRX").grid(row=0, column=2, padx=10, pady=10)
        # self.reconstructed_image_label2 = tk.Label(self.root, image=self.tk_img2)
        # self.reconstructed_image_label2.grid(row=1, column=2, padx=10, pady=10)

        # BER 顯示
        self.ber_label_EP = tk.Label(self.root, text="Now BER: 0.00e+00")
        self.ber_label_EP.grid(row=2, column=1, padx=10, pady=10)
        # self.ber_label_NRX = tk.Label(self.root, text="目前 BER: 0.00e+00")
        # self.ber_label_NRX.grid(row=2, column=2, padx=10, pady=10)
        
        self.avgber_label_EP = tk.Label(self.root, text="Avg BER: 0.00e+00")
        self.avgber_label_EP.grid(row=3, column=1, padx=10, pady=10)
        # self.avgber_label_NRX = tk.Label(self.root, text="平均 BER: 0.00e+00")
        # self.avgber_label_NRX.grid(row=3, column=2, padx=10, pady=10)
        
        # SNR 顯示
        self.snr_label = tk.Label(self.root, text="Now SNR: --", font=("Arial", 12))
        self.snr_label.grid(row=4, column=1, padx=10, pady=10, columnspan=2)
        # Throughput
        self.Throughput_label = tk.Label(self.root, text="Throughput: --", font=("Arial", 12))
        self.Throughput_label.grid(row=5, column=1, padx=10, pady=10, columnspan=2)


        # 控制按鈕
        self.start_button = tk.Button(self.root, text="begin", command=self.start_process)
        self.start_button.grid(row=7, column=1, pady=20)
        self.stop_button = tk.Button(self.root, text="end", command=self.root.destroy)
        self.stop_button.grid(row=7, column=2, pady=20)

    def start_process(self):
        """開始執行圖像更新"""
        if not self.running:
            self.running = True
            # self.update_images()
            threading.Thread(target=self.update_images, daemon=True).start()


    def update_images(self):
        """定期更新圖片、SNR、BER 和接收狀態"""
        # 進行 NRX 和 EP 檢測
        # NRX_picture, now_ber_NRX, SNR = self.NRX_DETECTOR(loaded_x_tim_dom=loaded_x_tim_dom, orign_bit=self.orign_bit)
        start_time = time.time()
        with self.lock:
            # EP_picture, now_ber_EP, SNR = self.EP_DETECTOR( iqtimes = self.iqtimes)
            EP_picture, now_ber_EP, SNR, Throughput, iqtimes = self.EP_DETECTOR(self.iqtimes)
            self.iqtimes = iqtimes
        
        end_time = time.time()  # 記錄結束時間
        elapsed_time = end_time - start_time  # 計算總執行時間
        # print(f"time {elapsed_time}")
        self.root.after(0, self.update_gui, EP_picture, now_ber_EP, SNR, Throughput)
        

    # 更新 Tkinter 圖片
    def update_gui(self, EP_picture, now_ber_EP, SNR, Throughput):
        received_correctly = True  # 先假設接收正確
        self.tk_img = ImageTk.PhotoImage(Image.fromarray(EP_picture).resize((250, 250), Image.LANCZOS))
        # self.tk_img2 = ImageTk.PhotoImage(Image.fromarray(NRX_picture).resize((250, 250), Image.LANCZOS))

        self.reconstructed_image_label.config(image=self.tk_img)
        # self.reconstructed_image_label2.config(image=self.tk_img2)

        # 記錄 BER 和 SNR 數據
        # self.ber_NRX_values.append(now_ber_NRX)
        self.ber_EP_values.append(now_ber_EP)
        self.snr_values.append(SNR if received_correctly else None)

        # 計算平均 BER
        # avg_ber_NRX = np.mean(self.ber_NRX_values)
        avg_ber_EP = np.mean(self.ber_EP_values)

        # 更新 BER、SNR 和狀態指示燈
        self.ber_label_EP.config(text=f"Now BER: {now_ber_EP:.2e}")
        # self.ber_label_NRX.config(text=f"目前 BER: {now_ber_NRX:.2e}")
        # self.avgber_label_NRX.config(text=f"平均 BER: {avg_ber_NRX:.2e}")
        self.avgber_label_EP.config(text=f"Avg BER: {avg_ber_EP:.2e}")

        # 更新 SNR 顯示
        self.snr_label.config(text=f"Now SNR: {SNR:.2f} dB" if received_correctly else "Now SNR: Unable to estimate")

        self.Throughput_label.config(text=f"Throughput: {Throughput / 1000000:.2f} Mbps")
        
        # 0.5 秒後自動更新
        self.root.after(5, self.update_images)


root = tk.Tk()
app = App(root)
root.mainloop()